# Query Classification: A / B / C / U

`baseline.ipynb`의 의료 범위 정의를 그대로 써서, 평가 세트 후보 질문을 LLM(Qwen2.5-7B-Instruct)으로 분류합니다.

| 라벨 | 의미 | 사용처 |
|---|---|---|
| A | 핵심 의료 | 의료 평가 세트 |
| B | 의료 인접 (영양, 운동, 비용, 보험, 수의학 등) | 별도 보관, 검색 성능만 참고 |
| C | 비의료 | 도메인 밖(거절해야 하는) 평가 세트 |
| U | 불확실 | 제외 |

플래그: `high_risk`(개인 의료 판단 요구), `time_sensitive`(바뀔 수 있는 가이드라인)

**분류 대상 (약 1.1만 개)**: 정답이 있는 질문만 대상으로 합니다.
- `med_url`: 정답 passage가 의료 도메인인 질문. validation 전체와 train 1,500개 샘플
- `no_med`: 의료 도메인 passage가 하나도 없는 질문. validation 전체와 train 1,000개 샘플

분류할 때는 질문과 함께 **데이터셋이 제공한 정답 passage**(없으면 첫 passage)를 보여줍니다. `stroke`처럼 질문만으로는 의미를 정할 수 없는 경우(Case 3)에 판단 근거로 쓰기 위해서입니다.

**검증**: Codex가 같은 기준(`labeling/LABELING_GUIDE.md`)으로 200개를 따로 라벨링하고, 마지막 섹션에서 LLM 라벨과의 일치율을 계산합니다.

**출력**: `data/labels/query_labels.parquet` → `LabeledBaseline.ipynb`에서 사용

## 0. 환경 설정

In [ ]:
%pip install -q -U datasets accelerate scikit-learn

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, re, json, random
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch

BASE_DIR = Path(os.environ.get("RAG_BASE_DIR", "/content/drive/MyDrive/RAG-Chat"))
LABEL_DIR = BASE_DIR / "data" / "labels"
LABEL_DIR.mkdir(parents=True, exist_ok=True)
LABEL_PATH = LABEL_DIR / "query_labels.parquet"
# Codex가 라벨링한 검수 시트 (repo의 Baseline/labeling/review_sample_labeled.csv를 이 위치에 업로드)
REVIEW_PATH = BASE_DIR / "labeling" / "review_sample_labeled.csv"

SEED = 42
DATASET_NAME, DATASET_VERSION = "microsoft/ms_marco", "v1.1"
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
N_TRAIN_MED = 1500     # train med_url 샘플 (의료 튜닝 세트용)
N_TRAIN_OOD = 1000     # train no_med 샘플 (도메인 밖 튜닝 세트용)
MAX_NEW_TOKENS = 48
GEN_BATCH = 32

random.seed(SEED)
np.random.seed(SEED)

## 1. 데이터와 후보 질문
코퍼스와 의료 도메인 판정은 `MedicalBaseline.ipynb`와 같은 코드를 씁니다.

In [ ]:
MEDICAL_DOMAINS = [
    "webmd.com", "webmd.boots.com", "mayoclinic.org", "mayoclinic.com", "medicinenet.com",
    "healthline.com", "medical-dictionary.thefreedictionary.com", "nih.gov", "medlineplus.gov",
    "cdc.gov", "everydayhealth.com", "nhs.uk", "medicalnewstoday.com", "rxlist.com",
    "medscape.com", "emedicinehealth.com", "wisegeekhealth.com", "health.howstuffworks.com",
    "cancer.org", "clevelandclinic.org", "patient.info", "hopkinsmedicine.org", "healthtap.com",
    "news-medical.net", "cancerresearchuk.org", "americanpregnancy.org", "spine-health.com",
    "who.int", "cancer.gov", "womenshealth.gov", "md-health.com", "heart.org", "healthhype.com",
    "familydoctor.org", "goodrx.com", "medhelp.org", "symptomfind.com", "medilexicon.com",
    "psychcentral.com", "healthcommunities.com", "med-health.net", "diabetes.org",
    "themedicalquestions.com", "health.harvard.edu", "drugs.com", "cancercenter.com",
    "rightdiagnosis.com", "innerbody.com", "labtestsonline.org", "sharecare.com", "umm.edu",
    "kidshealth.org", "health.com", "healthguideinfo.com", "medicalhealthtests.com",
]

def get_host(url):
    host = (urlparse(url).hostname or "").lower()
    return host[4:] if host.startswith("www.") else host

def is_medical_host(host):
    return any(host == d or host.endswith("." + d) for d in MEDICAL_DOMAINS)

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def has_answer(answers):
    return any(
        str(a).strip() and str(a).strip().lower().rstrip(".") != "no answer present"
        for a in (answers if answers is not None else [])
    )

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, name=DATASET_VERSION)
qa = {split: dataset[split].to_pandas() for split in ("train", "validation")}
for split, df in qa.items():
    print(split, len(df))

In [ ]:
rows = []
for split, df in qa.items():
    for qid, p in zip(df["query_id"], df["passages"]):
        for text, sel, url in zip(p["passage_text"], p["is_selected"], p["url"]):
            rows.append((split, int(qid), normalize_text(text), int(sel), url))

flat = pd.DataFrame(rows, columns=["split", "query_id", "text", "is_selected", "url"])
flat = flat[flat["text"] != ""]
flat["host"] = flat["url"].map(get_host)
host_is_med = {h: is_medical_host(h) for h in flat["host"].unique()}
flat["is_medical"] = flat["host"].map(host_is_med)

corpus = (
    flat[flat["is_medical"]]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
corpus.insert(0, "pid", np.arange(len(corpus)))   # pid == 행 번호 (인덱스 위치와 동일)
text2pid = dict(zip(corpus["text"], corpus["pid"]))

print(f"전체 passage: {len(flat):,} / 고유 텍스트: {flat['text'].nunique():,}")
print(f"의료 코퍼스: {len(corpus):,}")
print(corpus["host"].value_counts().head(10).to_string())

In [ ]:
def candidate_pool(split):
    df = qa[split]
    fs = flat[flat["split"] == split]
    selected_med = set(fs.loc[fs["is_medical"] & (fs["is_selected"] == 1), "query_id"])
    any_med = set(fs.loc[fs["is_medical"], "query_id"])
    # 분류용 문맥: 정답 passage 우선, 없으면 첫 passage
    context = (fs.sort_values("is_selected", ascending=False, kind="stable")
                 .drop_duplicates("query_id").set_index("query_id")["text"])

    base = pd.DataFrame({
        "query_id": df["query_id"].astype(int),
        "split": split,
        "query": df["query"].map(normalize_text),
        "query_type": df["query_type"],
        "answers": df["answers"].map(list),
    })
    base["pool"] = np.where(base["query_id"].isin(selected_med), "med_url",
                            np.where(base["query_id"].isin(any_med), None, "no_med"))
    base["context"] = base["query_id"].map(context).fillna("")
    return base[df["answers"].map(has_answer) & base["pool"].notna()].reset_index(drop=True)

val_pool, train_pool = candidate_pool("validation"), candidate_pool("train")
candidates = pd.concat([
    val_pool,
    train_pool[train_pool["pool"] == "med_url"].sample(N_TRAIN_MED, random_state=SEED),
    train_pool[train_pool["pool"] == "no_med"].sample(N_TRAIN_OOD, random_state=SEED),
], ignore_index=True)
print(candidates.groupby(["split", "pool"]).size())

## 2. LLM 분류

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype=torch.bfloat16, device_map="cuda")
llm.eval()

@torch.no_grad()
def generate(messages_list, batch_size=GEN_BATCH):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in messages_list]
    order = np.argsort([len(p) for p in prompts])          # 길이순 배치로 padding 낭비 줄이기
    outputs = [None] * len(prompts)
    for i in tqdm(range(0, len(prompts), batch_size)):
        idx = order[i:i + batch_size]
        enc = tokenizer([prompts[j] for j in idx], return_tensors="pt", padding=True).to(llm.device)
        out = llm.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           temperature=None, top_p=None, top_k=None)
        texts = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for j, t in zip(idx, texts):
            outputs[j] = t.strip()
    return outputs

In [ ]:
LABELS = ("A", "B", "C", "U")

SYSTEM_CLS = """You classify search questions by whether they ask for medical knowledge about human health.

Labels:
A = core medical: diseases, symptoms, diagnosis, medical tests, drugs, treatments and procedures, anatomy and physiology, prevention and infection, mental disorders.
B = medical-adjacent: nutrition, diet, exercise, healthy habits, general wellness, health insurance, healthcare costs, hospital operations, animal or veterinary health.
C = non-medical: the purpose of the question is not medical knowledge, even if a medical word appears (history of a medical invention, a person's biography, a non-medical meaning of a word, healthcare jobs and salaries).
U = uncertain: the question is too ambiguous to decide, even after reading the passage.

Flags (only meaningful for A or B):
high_risk = asks for a personal medical decision, such as stopping or changing a medication or choosing one's own dose.
time_sensitive = the answer depends on guidelines that may change, such as treatment recommendations, drug dosages or vaccine schedules.

Rules:
- Judge the purpose of the question, not keywords. A question with no medical term can be A, and one with a medical term can be C.
- If a word has several meanings (e.g. "stroke"), use the passage to decide. If the passage does not settle it, use U.
- Whether the passage contains the answer does not matter.

Examples:
"what causes your hands to shake" -> {"label": "A", "high_risk": false, "time_sensitive": false}
"who invented the mri machine" -> {"label": "C", "high_risk": false, "time_sensitive": false}
"how much water should a person drink daily" -> {"label": "B", "high_risk": false, "time_sensitive": false}
"what are the signs of severe dehydration" -> {"label": "A", "high_risk": false, "time_sensitive": false}
"should i stop taking my blood pressure medication" -> {"label": "A", "high_risk": true, "time_sensitive": false}
"what is the recommended treatment for hepatitis c" -> {"label": "A", "high_risk": false, "time_sensitive": true}
"how much does an mri cost" -> {"label": "B", "high_risk": false, "time_sensitive": false}
"where was dr. smith born" -> {"label": "C", "high_risk": false, "time_sensitive": false}

Output only one JSON object: {"label": "A|B|C|U", "high_risk": true|false, "time_sensitive": true|false}"""

def build_cls_messages(query, passage):
    user = f"Question: {query}\nPassage: {passage[:1000]}\n\nReturn the JSON only."
    return [{"role": "system", "content": SYSTEM_CLS}, {"role": "user", "content": user}]

def _as_bool(v):
    return v is True or str(v).strip().lower() == "true"

def parse_label(text):
    """LLM 출력 → label/flags. JSON 파싱에 실패하면 첫 A/B/C/U 글자를 쓰고, 그것도 없으면 U."""
    m = re.search(r"\{.*?\}", text, flags=re.S)
    try:
        d = json.loads(m.group(0))
        label = str(d.get("label", "")).strip().upper()
        if label in LABELS:
            return {"label": label, "high_risk": _as_bool(d.get("high_risk")),
                    "time_sensitive": _as_bool(d.get("time_sensitive")), "parse_ok": True}
    except (AttributeError, ValueError):
        pass
    m = re.search(r"\b([ABCU])\b", text)
    return {"label": m.group(1) if m else "U", "high_risk": False, "time_sensitive": False, "parse_ok": False}

def classify(queries, contexts):
    raw = generate([build_cls_messages(q, c) for q, c in zip(queries, contexts)])
    out = pd.DataFrame([parse_label(r) for r in raw])
    out["raw"] = raw
    return out

In [ ]:
print(build_cls_messages("what is a stroke", "A stroke occurs when blood flow to the brain is blocked.")[1]["content"])

In [ ]:
if LABEL_PATH.exists():
    labels = pd.read_parquet(LABEL_PATH)
    print("기존 라벨 로드:", len(labels))
else:
    pred = classify(candidates["query"].tolist(), candidates["context"].tolist())
    labels = pd.concat([candidates.reset_index(drop=True), pred], axis=1)
    labels.to_parquet(LABEL_PATH, index=False)
print("JSON 파싱 실패:", (~labels["parse_ok"]).sum())

## 3. 분류 결과 분포

In [ ]:
display(pd.crosstab([labels["split"], labels["pool"]], labels["label"], margins=True))
display(pd.crosstab(labels["pool"], labels["label"], normalize="index").round(3))
print("high_risk:", labels["high_risk"].sum(), "/ time_sensitive:", labels["time_sensitive"].sum())

pd.set_option("display.max_colwidth", 120)
for (pool, lab), g in labels.groupby(["pool", "label"]):
    print(f"\n=== {pool} / {lab} ({len(g)}) ===")
    print("\n".join(g.sample(min(8, len(g)), random_state=SEED)["query"]))

## 4. Codex 라벨과 비교 (검증)

1. Codex가 `Baseline/labeling/review_sample.csv`를 가이드대로 채워 `review_sample_labeled.csv`로 저장합니다.
2. 그 파일을 Drive의 `RAG-Chat/labeling/`에 업로드합니다.
3. 아래 셀에서 정확도, Cohen's κ, 혼동행렬, 불일치 사례를 확인합니다.

검수 시트는 validation 후보에서 뽑았으므로(`med_url` 120개, `no_med` 80개) 모두 위 분류 대상에 들어 있습니다.

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report, confusion_matrix

if not REVIEW_PATH.exists():
    print("검수 파일이 아직 없습니다:", REVIEW_PATH)
else:
    review = pd.read_csv(REVIEW_PATH)
    review["label"] = review["label"].astype(str).str.strip().str.upper()
    merged = review.merge(labels[["query_id", "pool", "label", "high_risk", "time_sensitive"]],
                          on="query_id", suffixes=("_codex", "_llm"))
    print(f"매칭: {len(merged)} / {len(review)}")
    y_true, y_pred = merged["label_codex"], merged["label_llm"]
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}   Cohen's kappa: {cohen_kappa_score(y_true, y_pred):.3f}")
    print(classification_report(y_true, y_pred, labels=list(LABELS), zero_division=0))
    display(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=list(LABELS)),
                         index=[f"codex_{l}" for l in LABELS], columns=[f"llm_{l}" for l in LABELS]))

    # 실제로 쓰는 구분(A vs C)에서의 일치도: 의료 평가 세트와 도메인 밖 세트가 얼마나 정확한지
    ac = merged[y_true.isin(["A", "C"]) | y_pred.isin(["A", "C"])]
    print(f"A/C 관련 {len(ac)}건 중 일치 {(ac['label_codex'] == ac['label_llm']).mean():.3f}")

    disagree = merged[y_true != y_pred][["query", "pool", "label_codex", "label_llm", "note"]]
    disagree.to_csv(LABEL_DIR / "review_disagreements.csv", index=False, encoding="utf-8-sig")
    display(disagree)